# c-TF-IDF: ejemplo con conteos reales

Apoyo para la sección 3.6. Reconstruimos **solo la ponderación léxica** de la representación base guardada, sobre las mismas 1.000 reseñas y sus asignaciones. No calculamos embeddings, agrupamientos, sentimientos ni respuestas LLM.

Los grupos 0 y 5 se eligen para ilustrar un contraste de vocabulario y tamaño. No son una muestra para evaluar calidad. Los pesos c-TF-IDF reconstruidos **no son los pesos finales de KeyBERTInspired/MMR** del artefacto.

La libreta verifica el artefacto con el lector del proyecto y conserva hashes de procedencia. Sus salidas y la exportación pueden leerse sin acceso al dataset; volver a ejecutarla requiere las copias locales en `data/`.

In [1]:
from pathlib import Path
import os, json, hashlib, importlib.metadata
import numpy as np
import pandas as pd
from IPython.display import display

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "config/storage-difusion.toml").is_file())
os.environ["VOC_DATA_DIR"] = str(ROOT / "data")
os.environ["VOC_ANALYSIS_READ_DIRS"] = str(ROOT / "data")
from voc.analysis.store import read_analysis
from voc_ml.topics.bertopic import STOP_WORDS
from sklearn.feature_extraction.text import CountVectorizer
from bertopic import BERTopic
from bertopic.vectorizers import ClassTfidfTransformer

analysis_id = "85e5ea66-a176-44e3-9d67-dba87090f98f"
folder = ROOT / "data/analyses" / analysis_id
saved = read_analysis(analysis_id)
manifest = json.loads((folder / "manifest.json").read_text())
config = manifest["configuration"]["representation"]
assert (config["ngram_min"], config["ngram_max"], config["min_df"], config["reduce_frequent_words"]) == (1, 2, 1, True)
assignments = pd.DataFrame([{"record_id": a.record_id, "topic_id": a.topic_id} for a in saved.assignments])
reviews = saved.reviews.merge(assignments, on="record_id", validate="one_to_one").sort_values("vector_row")
assert len(reviews) == 1000
print(f"Artefacto verificado: {len(reviews)} reseñas; {reviews.topic_id.nunique()-1} tópicos; {(reviews.topic_id == -1).sum()} sin asignar.")

/workspace/vaults/sonar/code/repos/voc-difusion/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Artefacto verificado: 1000 reseñas; 18 tópicos; 47 sin asignar.


## Del texto a la matriz de conteos

Agregamos las reseñas por ID de grupo en el orden conservado, incluidos los textos sin asignación (`-1`) como un documento agregado, tal como hace la implementación. Aplicamos la preparación multilingüe de BERTopic, la lista de exclusiones del proyecto y `CountVectorizer(ngram_range=(1,2), min_df=1)`.

Cada celda cuenta **apariciones de un término**, no reseñas. Los bigramas se calculan sobre el documento concatenado de cada grupo; pueden cruzar límites entre reseñas, una propiedad de esta preparación. No interpretamos estos conteos como frecuencia documental.

In [2]:
topic_ids = sorted(reviews.topic_id.unique().tolist())
grouped = [" ".join(reviews.loc[reviews.topic_id == t, "model_text"]) for t in topic_ids]
cleaned = BERTopic(language="multilingual")._preprocess_text(grouped)
vectorizer = CountVectorizer(stop_words=list(STOP_WORDS), ngram_range=(1, 2), min_df=1)
X = vectorizer.fit_transform(cleaned)
terms = vectorizer.get_feature_names_out()
f = np.asarray(X.sum(axis=0)).ravel()
N = np.asarray(X.sum(axis=1)).ravel()
A = int(N.mean())  # misma conversión entera que ClassTfidfTransformer
weights = ClassTfidfTransformer(reduce_frequent_words=True).fit_transform(X.copy())

selected_terms = ["aplicación", "fácil", "usar", "fácil usar", "no abre"]
rows = []
for term in selected_terms:
    j = vectorizer.vocabulary_[term]
    row = {"termino": term, "grupo_0": int(X[topic_ids.index(0), j]),
           "grupo_5": int(X[topic_ids.index(5), j]), "corpus_f": int(f[j])}
    for t in (0, 5):
        i = topic_ids.index(t)
        value = float(np.sqrt(X[i, j] / N[i]) * np.log1p(A / f[j]))
        assert np.isclose(value, weights[i, j])
        row[f"peso_grupo_{t}"] = value
    rows.append(row)
table = pd.DataFrame(rows)
display(table)
print(f"Vocabulario: {len(terms)} términos; conteos totales: {int(N.sum())}; documentos agregados: {len(topic_ids)}; A = {A}.")
for t in (0, 5):
    print(f"Grupo {t}: {(reviews.topic_id == t).sum()} reseñas; N(c) = {N[topic_ids.index(t)]}.")

,termino,grupo_0,grupo_5,corpus_f,peso_grupo_0,peso_grupo_5
0,aplicación,158,2,216,0.175185,0.151496
1,fácil,10,22,47,0.079548,0.906900
2,usar,21,15,47,0.115277,0.748848
3,fácil usar,3,10,19,0.056007,0.785952
4,no abre,53,0,53,0.176330,0.000000


Vocabulario: 8901 términos; conteos totales: 18457; documentos agregados: 19; A = 971.
Grupo 0: 529 reseñas; N(c) = 14947.
Grupo 5: 30 reseñas; N(c) = 253.


## Un peso calculado paso a paso

Con `reduce_frequent_words=True`, el peso es

\[w(t,c)=\sqrt{n(t,c)/N(c)}\;\ln(1+A/f(t)).\]

`N(c)` incluye todo el vocabulario del grupo, no solo los términos de la tabla. `f(t)` suma sus apariciones en todos los documentos agregados, incluido `-1`. `A` es la longitud media de esos documentos convertida a entero. No es una probabilidad.

La tabla muestra que «fácil» ocupa una proporción mayor del vocabulario del grupo 5 que del grupo 0. Esto explica una ponderación distinta, sin probar que alguno de los grupos sea mejor.

In [3]:
example = table.loc[table.termino == "fácil"].iloc[0]
print(f"fácil, grupo 5: sqrt({example.grupo_5}/{N[topic_ids.index(5)]}) * ln(1 + {A}/{example.corpus_f}) = {example.peso_grupo_5:.6f}")

fácil, grupo 5: sqrt(22/253) * ln(1 + 971/47) = 0.906900


## Exportación para el reporte

Se guarda una exportación determinista de esta reconstrucción, identificada por el hash de su contenido. Incluye conteos, pesos, parámetros, versiones y hashes de entrada. No incluye textos ni credenciales. No modifica el análisis de origen.

In [4]:
evidence = {
    "schema_version": "voc-report-ctfidf-example-v1", "analysis_id": analysis_id,
    "manifest_sha256": hashlib.sha256((folder / "manifest.json").read_bytes()).hexdigest(),
    "database_sha256": hashlib.sha256((folder / "analysis.duckdb").read_bytes()).hexdigest(),
    "rows": len(reviews), "grouped_documents": len(topic_ids), "includes_unassigned": True,
    "vocabulary_size": len(terms), "total_counts": int(N.sum()), "A": A,
    "groups": {str(t): {"reviews": int((reviews.topic_id == t).sum()), "N": int(N[topic_ids.index(t)])} for t in (0, 5)},
    "lexical_policy": {"ngram_range": [1, 2], "min_df": 1, "stop_words": list(STOP_WORDS), "reduce_frequent_words": True},
    "software": {p: importlib.metadata.version(p) for p in ["bertopic", "scikit-learn", "numpy", "pandas"]},
    "selection": "Grupos 0 y 5 elegidos para ilustrar contraste; no evaluación de calidad.",
    "scope": "Reconstrucción c-TF-IDF, no pesos finales KeyBERTInspired/MMR.",
    "counts": table.to_dict(orient="records"),
}
payload = json.dumps(evidence, ensure_ascii=False, indent=2) + "\n"
digest = hashlib.sha256(payload.encode()).hexdigest()
export = ROOT / "reports/drafts/difusion/exports/ejemplo-ctfidf" / digest[:12]
export.mkdir(parents=True, exist_ok=True)
(export / "conteos.json").write_text(payload)
table.to_csv(export / "conteos.csv", index=False)
print(f"Exportación: {export.relative_to(ROOT)}; SHA-256: {digest}")

Exportación: reports/difusion/exports/ejemplo-ctfidf/ef4f51ba5a08; SHA-256: ef4f51ba5a080e22d5492c15e42e6ae90e9b43506a0533dab2bd62dd3c25ef99
